In [1]:
!pip install -q langchain-mistralai langgraph python-dotenv

In [2]:
from langgraph.store.memory import InMemoryStore
from langchain_mistralai import MistralAIEmbeddings

In [3]:
from google.colab import userdata
import os
os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

In [4]:
embeddings = MistralAIEmbeddings(model="mistral-embed")

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

In [5]:
# THE KEY DIFFERENCE from a bare InMemoryStore(): passing an `index` config
# tells the store to embed every value's "text" field automatically on
# put(), and enables SEMANTIC search on top of the usual exact-namespace
# lookup. Without this, store.search() with a natural-language `query` would
# have no way to rank results by meaning.
store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1024,          # mistral-embed's output vector size
        "fields": ["text"],    # which field(s) of each stored value get embedded
    }
)

In [6]:
user_id = "1"
namespace = ("memories", user_id)

In [7]:
store.put(namespace, "1", {"text": "The user likes the color blue."})
store.put(namespace, "2", {"text": "The user is learning LangGraph and LangChain."})
store.put(namespace, "3", {"text": "The user enjoys hiking on weekends."})
store.put(namespace, "4", {"text": "The user works as a software engineer."})

In [8]:
store.search(namespace)

[Item(namespace=['memories', '1'], key='1', value={'text': 'The user likes the color blue.'}, created_at='2026-09-23T07:06:20.185720+00:00', updated_at='2026-09-23T07:06:20.185724+00:00', score=None),
 Item(namespace=['memories', '1'], key='2', value={'text': 'The user is learning LangGraph and LangChain.'}, created_at='2026-09-23T07:06:20.441794+00:00', updated_at='2026-09-23T07:06:20.441799+00:00', score=None),
 Item(namespace=['memories', '1'], key='3', value={'text': 'The user enjoys hiking on weekends.'}, created_at='2026-09-23T07:06:20.602694+00:00', updated_at='2026-09-23T07:06:20.602699+00:00', score=None),
 Item(namespace=['memories', '1'], key='4', value={'text': 'The user works as a software engineer.'}, created_at='2026-09-23T07:06:20.761620+00:00', updated_at='2026-09-23T07:06:20.761624+00:00', score=None)]

In [9]:
# THE PAYOFF: query is embedded the same way stored memories were, and
# results are ranked by SIMILARITY -- meaning-based retrieval, exactly like
# the FAISS/Chroma vector stores from the RAG labs, but built directly into
# the store used for LangGraph's long-term memory.
results = store.search(namespace, query="What does the user do for a living?")

for r in results:
    print(r.value["text"], "-> score:", r.score)

The user works as a software engineer. -> score: 0.8334068052773848
The user likes the color blue. -> score: 0.7899197524476214
The user enjoys hiking on weekends. -> score: 0.7649754891398871
The user is learning LangGraph and LangChain. -> score: 0.7498845677486844


In [10]:
results = store.search(namespace, query="What are the user's hobbies?")

for r in results:
    print(r.value["text"], "-> score:", r.score)

The user works as a software engineer. -> score: 0.7875642615760226
The user enjoys hiking on weekends. -> score: 0.7737987256264884
The user likes the color blue. -> score: 0.7714539388505667
The user is learning LangGraph and LangChain. -> score: 0.7192289853992483


In [11]:
# Only the single closest match, instead of all 4 ranked.
top_result = store.search(namespace, query="What color does the user like?", limit=1)
top_result

[Item(namespace=['memories', '1'], key='1', value={'text': 'The user likes the color blue.'}, created_at='2026-09-23T07:06:20.185720+00:00', updated_at='2026-09-23T07:06:20.185724+00:00', score=0.8764277218645569)]